In [1]:
!pip install -q rapidfuzz

import os
import time
import requests
import json
import urllib.request
import zipfile
from collections import Counter
import itertools

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from rapidfuzz import process, fuzz
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware Engine: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 23.3 MB/s eta 0:00:00
Mounted at /content/drive
Hardware Engine: cpu


In [2]:
def download_bulk_fda_data(target_records=100000):
    SAVE_PATH = "/content/drive/MyDrive/FAERS_PROJECT/raw_data_bulk.parquet"

    print("1. Fetching list of bulk download files from FDA...")
    downloads_url = "https://api.fda.gov/download.json"
    resp = requests.get(downloads_url)

    if resp.status_code != 200:
        print("Error accessing FDA bulk system.")
        return None

    data = resp.json()
    partitions = data['results']['drug']['event']['partitions']

    total_records = 0
    downloaded_files = []
    os.makedirs("fda_temp", exist_ok=True)

    print("2. Starting high-speed downloads...")
    for part in partitions:
        if total_records >= target_records: break

        file_url = part['file']
        file_name = file_url.split('/')[-1]
        local_zip_path = f"fda_temp/{file_name}"

        print(f" -> Downloading {file_name} (Contains {part['records']} records)...")
        urllib.request.urlretrieve(file_url, local_zip_path)
        downloaded_files.append(local_zip_path)
        total_records += int(part['records'])

    all_data = []
    print("\n3. Extracting and Parsing JSON files...")
    for zip_path in downloaded_files:
        with zipfile.ZipFile(zip_path, 'r') as z:
            json_filename = z.namelist()[0]
            with z.open(json_filename) as f:
                json_data = json.loads(f.read())
                for entry in json_data['results']:
                    patient = entry.get("patient", {})
                    all_data.append({
                        "report_id": entry.get("safetyreportid"),
                        "age": patient.get("patientonsetage"),
                        "gender": patient.get("patientsex"),
                        "drugs": ", ".join([d.get("medicinalproduct", "") for d in patient.get("drug", [])]),
                        "reactions": ", ".join([r.get("reactionmeddrapt", "") for r in patient.get("reaction", [])])
                    })

    print("\n4. Converting to DataFrame and Saving to Parquet...")
    df = pd.DataFrame(all_data)
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    df.to_parquet(SAVE_PATH)
    print(f"Mission Accomplished! {len(df)} records safely stored at {SAVE_PATH}")
    return df

df_bulk = download_bulk_fda_data(target_records=100000)

1. Fetching list of bulk download files from FDA...
2. Starting high-speed downloads...
 -> Downloading drug-event-0001-of-0005.json.zip (Contains 12000 records)...
 -> Downloading drug-event-0002-of-0005.json.zip (Contains 12000 records)...
 -> Downloading drug-event-0003-of-0005.json.zip (Contains 12000 records)...
 -> Downloading drug-event-0004-of-0005.json.zip (Contains 12000 records)...
 -> Downloading drug-event-0005-of-0005.json.zip (Contains 6071 records)...
 -> Downloading drug-event-0001-of-0029.json.zip (Contains 12000 records)...
 -> Downloading drug-event-0002-of-0029.json.zip (Contains 12000 records)...
 -> Downloading drug-event-0003-of-0029.json.zip (Contains 12000 records)...
 -> Downloading drug-event-0004-of-0029.json.zip (Contains 12000 records)...

3. Extracting and Parsing JSON files...

4. Converting to DataFrame and Saving to Parquet...
Mission Accomplished! 102071 records safely stored at /content/drive/MyDrive/FAERS_PROJECT/raw_data_bulk.parquet


In [4]:
import os
import sys
import shutil
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Initialize Spark Session
print("Igniting Spark Engine for Deep NLP Cleaning... 🚀")
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("FAERS_Deep_Cleaning") \
    .config("spark.driver.memory", "8g") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

# Load raw dataset
raw_path = "/content/drive/MyDrive/FAERS_PROJECT/raw_data_bulk.parquet"
df_spark = spark.read.parquet(raw_path)

print(f"Initial records: {df_spark.count()}")

# Specialized Regex Cleaning for Drugs
clean_regex_drugs = df_spark.withColumn(
    "drugs_nlp_clean",
    F.lower(F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(F.col("drugs"), r"\(.*?\)", ""),
                    r"\b\d+(\.\d+)?\s*(mg|g|ml|mcg|m2|%|iu|units)\b", ""
                ),
                r"[^a-zA-Z,\s]", ""
            ),
            r"\s+", " "
        )
    ))
)

# Standardize Reactions
df_final_clean = clean_regex_drugs.withColumn(
    "reactions_nlp_clean",
    F.lower(F.trim(F.col("reactions")))
)

# Filter empty records post-cleaning
df_final_clean = df_final_clean.filter(
    (F.col("drugs_nlp_clean") != "") &
    (F.col("reactions_nlp_clean") != "") &
    (F.col("drugs_nlp_clean").isNotNull())
)

# Output Path Handling
output_path = "/content/drive/MyDrive/FAERS_PROJECT/pytorch_ready_data.parquet"

if os.path.exists(output_path):
    print(f"🗑️ Purging old structure at {output_path}")
    shutil.rmtree(output_path)

# Save Purified Dataset
print(f"Writing purified dataset to: {output_path}...")
df_final_clean.write.mode("overwrite").parquet(output_path)

print(f"✅ Phase 2 Complete. Cleaned records: {df_final_clean.count()}")

Igniting Spark Engine for Deep NLP Cleaning... 🚀
Initial records: 102071
🗑️ Purging old structure at /content/drive/MyDrive/FAERS_PROJECT/pytorch_ready_data.parquet
Writing purified dataset to: /content/drive/MyDrive/FAERS_PROJECT/pytorch_ready_data.parquet...
✅ Phase 2 Complete. Cleaned records: 102070


In [6]:
import urllib.request
import zipfile
import os
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm

print("1. Loading raw dataset...")
df = pd.read_parquet("/content/drive/MyDrive/FAERS_PROJECT/pytorch_ready_data.parquet")
all_drugs_list = df['drugs_nlp_clean'].str.split(',').explode().str.strip().unique()
all_drugs_list = [d for d in all_drugs_list if d and d != '']
print(f"Total unique raw drug strings to process: {len(all_drugs_list)}")

print("\n2. Downloading OFFICIAL FDA National Drug Code (NDC) database...")

# Updated URL and added User-Agent to prevent 404/Forbidden errors
url = "https://www.accessdata.fda.gov/cder/ndctext.zip"
opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')]
urllib.request.install_opener(opener)

try:
    urllib.request.urlretrieve(url, "ndc.zip")
    with zipfile.ZipFile("ndc.zip", 'r') as zip_ref:
        zip_ref.extractall("ndc_data")
    print("✅ NDC Database downloaded and extracted.")
except Exception as e:
    print(f"❌ Download failed: {e}")
    # Fallback to OpenFDA API method if the zip link is down
    print("Attempting API fallback...")

# Load product data
ndc_df = pd.read_csv("ndc_data/product.txt", sep='\t', encoding='latin1', on_bad_lines='skip')
generics_raw = ndc_df['NONPROPRIETARYNAME'].dropna().str.lower().unique()

expanded_generics = []
for g in generics_raw:
    expanded_generics.extend([str(x).strip() for x in str(g).split(';')])

STANDARD_GENERICS = list(set(expanded_generics))
print(f"✅ Loaded {len(STANDARD_GENERICS)} unique official generic names from FDA NDC.")

print("\n3. Running Entity Resolution Engine (Fuzzy Matching)...")
mapping_dict = {}
for dirty_drug in tqdm(all_drugs_list, desc="Standardizing"):
    best_match = process.extractOne(dirty_drug, STANDARD_GENERICS, scorer=fuzz.WRatio)

    if best_match and best_match[1] >= 85:
        mapping_dict[dirty_drug] = best_match[0]
    else:
        mapping_dict[dirty_drug] = dirty_drug

def apply_semantic_mapping(drug_string):
    if not isinstance(drug_string, str): return ""
    drugs = [d.strip() for d in drug_string.split(',')]
    mapped_drugs = [mapping_dict.get(d, d) for d in drugs]
    return ", ".join(list(set(mapped_drugs)))

df['drugs_standardized'] = df['drugs_nlp_clean'].apply(apply_semantic_mapping)
SAVE_PATH = "/content/drive/MyDrive/FAERS_PROJECT/pytorch_ready_data_standardized.parquet"
df.to_parquet(SAVE_PATH)
print(f"\n🎯 Master standardized dataset saved to: {SAVE_PATH}")

1. Loading raw dataset...
Total unique raw drug strings to process: 26334

2. Downloading OFFICIAL FDA National Drug Code (NDC) database...
✅ NDC Database downloaded and extracted.
✅ Loaded 15553 unique official generic names from FDA NDC.

3. Running Entity Resolution Engine (Fuzzy Matching)...


Standardizing: 100%|██████████| 26334/26334 [16:23<00:00, 26.77it/s]



🎯 Master standardized dataset saved to: /content/drive/MyDrive/FAERS_PROJECT/pytorch_ready_data_standardized.parquet


In [7]:
print("1. Rebuilding target architecture to 21 Clinical Phenotypes...")
df = pd.read_parquet("/content/drive/MyDrive/FAERS_PROJECT/pytorch_ready_data_standardized.parquet")
df['raw_reaction_list'] = df['reactions_nlp_clean'].apply(lambda x: [r.strip() for r in x.split(',') if r.strip()])

CLINICAL_PHENOTYPES = {
    'gastrointestinal haemorrhage': 'Risk: GI Bleeding', 'haemorrhage': 'Risk: Systemic Bleeding',
    'epistaxis': 'Risk: Systemic Bleeding', 'melaena': 'Risk: GI Bleeding',
    'haematemesis': 'Risk: GI Bleeding', 'contusion': 'Risk: Bruising/Bleeding',
    'atrial fibrillation': 'Risk: Arrhythmia', 'tachycardia': 'Risk: Arrhythmia',
    'bradycardia': 'Risk: Arrhythmia', 'myocardial infarction': 'Risk: Ischemic Heart Disease',
    'angina pectoris': 'Risk: Ischemic Heart Disease', 'cardiac failure': 'Risk: Heart Failure',
    'somnolence': 'Risk: CNS Depression', 'dizziness': 'Risk: Vestibular/CNS',
    'confusional state': 'Risk: Cognitive Impairment', 'tremor': 'Risk: Extrapyramidal/Movement',
    'seizure': 'Risk: Seizure', 'headache': 'Risk: Headache/Migraine',
    'renal failure': 'Risk: Acute Renal Toxicity', 'acute kidney injury': 'Risk: Acute Renal Toxicity',
    'hepatic failure': 'Risk: Hepatotoxicity', 'hepatotoxicity': 'Risk: Hepatotoxicity',
    'liver function test abnormal': 'Risk: Hepatotoxicity',
    'dyspnoea': 'Risk: Dyspnea/Bronchospasm', 'pneumonia': 'Risk: Respiratory Infection',
    'hypoglycaemia': 'Risk: Hypoglycemia', 'hyperglycaemia': 'Risk: Hyperglycemia',
    'hypokalaemia': 'Risk: Electrolyte Imbalance',
    'anaphylactic reaction': 'Risk: Severe Anaphylaxis', 'angiooedema': 'Risk: Severe Anaphylaxis',
    'rash': 'Risk: Mild Dermatological', 'pruritus': 'Risk: Mild Dermatological'
}

VAGUE_NOISE = {'nausea', 'vomiting', 'fatigue', 'pain', 'asthenia', 'malaise', 'feeling abnormal',
               'pyrexia', 'condition aggravated', 'drug ineffective', 'death', 'off label use'}

def extract_phenotypes(reaction_list):
    phenotypes = set()
    for r in reaction_list:
        if r in VAGUE_NOISE: continue
        if r in CLINICAL_PHENOTYPES: phenotypes.add(CLINICAL_PHENOTYPES[r])
    return list(phenotypes)

df['phenotype_targets'] = df['raw_reaction_list'].apply(extract_phenotypes)
df_phenotypes = df[df['phenotype_targets'].map(len) > 0].reset_index(drop=True)

SAVE_PATH = "/content/drive/MyDrive/FAERS_PROJECT/pytorch_phenotype_targets.parquet"
df_phenotypes.to_parquet(SAVE_PATH)
print(f"Surgical precision dataset saved to: {SAVE_PATH}")

1. Rebuilding target architecture to 21 Clinical Phenotypes...
Surgical precision dataset saved to: /content/drive/MyDrive/FAERS_PROJECT/pytorch_phenotype_targets.parquet


In [8]:
print("1. Loading the current phenotype dataset...")
df = pd.read_parquet("/content/drive/MyDrive/FAERS_PROJECT/pytorch_phenotype_targets.parquet")

# FILTER 1: Max 6 drugs
df['drug_count'] = df['drugs_standardized'].apply(lambda x: len([d for d in x.split(',') if d.strip()]))
df_clean = df[df['drug_count'] <= 6].copy()

# FILTER 2: Max 3 specific severe phenotypes
df_clean['phenotype_count'] = df_clean['phenotype_targets'].apply(len)
df_clean = df_clean[df_clean['phenotype_count'] <= 3].copy()

# FILTER 3: Drop ultra-rare drugs (< 50 reports)
all_clean_drugs = list(itertools.chain.from_iterable(df_clean['drugs_standardized'].apply(lambda x: [d.strip() for d in x.split(',') if d.strip()])))
valid_drugs = {drug for drug, count in Counter(all_clean_drugs).items() if count >= 50}

def filter_rare_drugs(drug_string):
    drugs = [d.strip() for d in drug_string.split(',')]
    valid = [d for d in drugs if d in valid_drugs]
    return ", ".join(valid)

df_clean['drugs_purified'] = df_clean['drugs_standardized'].apply(filter_rare_drugs)
df_clean = df_clean[df_clean['drugs_purified'] != ""].reset_index(drop=True)

SAVE_PATH = "/content/drive/MyDrive/FAERS_PROJECT/pytorch_phenotype_purified.parquet"
df_clean.to_parquet(SAVE_PATH)
print(f"Data Purification Complete. Purified dataset saved to: {SAVE_PATH}")

1. Loading the current phenotype dataset...
Data Purification Complete. Purified dataset saved to: /content/drive/MyDrive/FAERS_PROJECT/pytorch_phenotype_purified.parquet


In [16]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import time
import joblib

# 1. Named tokenizer function to avoid PicklingError
def drug_tokenizer(text):
    return [d.strip() for d in text.split(',') if d.strip()]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware Engine: {device}")

# 2. Data Loading
df_clean = pd.read_parquet("/content/drive/MyDrive/FAERS_PROJECT/pytorch_phenotype_purified.parquet")

# 3. Target and Feature Preparation
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df_clean['phenotype_targets']).astype(np.float32)
PHENOTYPES = list(mlb.classes_)

tfidf = TfidfVectorizer(
    max_features=3000,
    lowercase=False,
    tokenizer=drug_tokenizer,
    token_pattern=None
)
X = tfidf.fit_transform(df_clean['drugs_purified']).astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42)

class PurifiedDataset(Dataset):
    def __init__(self, X_sparse, y_dense):
        self.X, self.y = X_sparse, y_dense
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx].toarray().squeeze(), dtype=torch.float32), \
               torch.tensor(self.y[idx], dtype=torch.float32)

train_loader = DataLoader(PurifiedDataset(X_train, y_train), batch_size=256, shuffle=True)
val_loader = DataLoader(PurifiedDataset(X_val, y_val), batch_size=256, shuffle=False)

# 4. Neural Network Architecture
class FinalSafetyPredictor(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, out_dim)
        )
    def forward(self, x):
        return self.network(x)

model = FinalSafetyPredictor(X.shape[1], y.shape[1]).to(device)

# 5. Training Setup
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

pos_weight = torch.clamp(((y_train_tensor.size(0) - y_train_tensor.sum(dim=0)) / (y_train_tensor.sum(dim=0) + 1e-5)), max=5.0).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.05)

print("\n--- Starting Final Training ---")
EPOCHS = 12
for epoch in range(EPOCHS):
    start_time = time.time()
    model.train()
    total_loss = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# 6. Dynamic Threshold Optimization
print("\n--- Optimizing Thresholds ---")
model.eval()
all_probs, all_targets = [], []
with torch.no_grad():
    for X_b, y_b in val_loader:
        outputs = model(X_b.to(device))
        all_probs.append(torch.sigmoid(outputs).cpu().numpy())
        all_targets.append(y_b.numpy())

y_true = np.vstack(all_targets)
y_probs = np.vstack(all_probs)
best_thresholds = []

for i in range(y_true.shape[1]):
    best_f1, best_t = 0, 0.35
    for t in np.arange(0.1, 0.9, 0.05):
        score = f1_score(y_true[:, i], (y_probs[:, i] > t).astype(int), zero_division=0)
        if score > best_f1: best_f1, best_t = score, t
    best_thresholds.append(best_t)

# 7. Saving All Artifacts
torch.save(model.state_dict(), "/content/drive/MyDrive/FAERS_PROJECT/final_purified_model.pth")
np.save("/content/drive/MyDrive/FAERS_PROJECT/final_purified_thresholds.npy", best_thresholds)
joblib.dump(tfidf, "/content/drive/MyDrive/FAERS_PROJECT/tfidf_vectorizer.pkl")
joblib.dump(mlb, "/content/drive/MyDrive/FAERS_PROJECT/mlb_binarizer.pkl")

print("\n✅ Phase 6 Complete: All artifacts saved successfully.")

Hardware Engine: cpu

--- Starting Final Training ---
Epoch 01/12 | Loss: 0.5879
Epoch 02/12 | Loss: 0.5086
Epoch 03/12 | Loss: 0.4937
Epoch 04/12 | Loss: 0.4867
Epoch 05/12 | Loss: 0.4810
Epoch 06/12 | Loss: 0.4734
Epoch 07/12 | Loss: 0.4705
Epoch 08/12 | Loss: 0.4664
Epoch 09/12 | Loss: 0.4646
Epoch 10/12 | Loss: 0.4627
Epoch 11/12 | Loss: 0.4607
Epoch 12/12 | Loss: 0.4563

--- Optimizing Thresholds ---

✅ Phase 6 Complete: All artifacts saved successfully.


In [17]:
import torch
import numpy as np
import torch.nn as nn
import joblib

# Required for joblib to successfully load the TF-IDF vectorizer
def drug_tokenizer(text):
    return [d.strip() for d in text.split(',') if d.strip()]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Load All Saved Artifacts ---
try:
    tfidf = joblib.load("/content/drive/MyDrive/FAERS_PROJECT/tfidf_vectorizer.pkl")
    mlb = joblib.load("/content/drive/MyDrive/FAERS_PROJECT/mlb_binarizer.pkl")
    optimal_thresholds = np.load("/content/drive/MyDrive/FAERS_PROJECT/final_purified_thresholds.npy")

    input_dim = len(tfidf.vocabulary_)
    PHENOTYPES = list(mlb.classes_)

    class FinalSafetyPredictor(nn.Module):
        def __init__(self, in_dim, out_dim):
            super().__init__()
            self.network = nn.Sequential(
                nn.Linear(in_dim, 512),
                nn.BatchNorm1d(512),
                nn.ReLU(),
                nn.Dropout(0.5),
                nn.Linear(512, out_dim)
            )
        def forward(self, x):
            return self.network(x)

    model = FinalSafetyPredictor(input_dim, len(PHENOTYPES)).to(device)
    model.load_state_dict(torch.load("/content/drive/MyDrive/FAERS_PROJECT/final_purified_model.pth"))
    model.eval()
    print("✅ Model, Thresholds, and Transformers Loaded Successfully!\n")
except Exception as e:
    print(f"⚠️ Load Error: {e}")
    raise e

# --- 2. Clinical Knowledge Mapping ---
HIERARCHY_MAP = {
    '🫀 Cardiovascular Disorders': ['Risk: Arrhythmia', 'Risk: Heart Failure', 'Risk: Ischemic Heart Disease'],
    '🧠 Nervous System & Psych': ['Risk: CNS Depression', 'Risk: Cognitive Impairment', 'Risk: Extrapyramidal/Movement', 'Risk: Headache/Migraine', 'Risk: Seizure', 'Risk: Vestibular/CNS'],
    '🩸 Bleeding & GI Disorders': ['Risk: GI Bleeding', 'Risk: Systemic Bleeding', 'Risk: Bruising/Bleeding'],
    '🫁 Respiratory Disorders': ['Risk: Dyspnea/Bronchospasm', 'Risk: Respiratory Infection'],
    '🧪 Hepatic & Renal Disorders': ['Risk: Acute Renal Toxicity', 'Risk: Hepatotoxicity'],
    '🔬 Metabolic & Endocrine': ['Risk: Electrolyte Imbalance', 'Risk: Hyperglycemia', 'Risk: Hypoglycemia'],
    '⚠️ Immune & Skin Disorders': ['Risk: Mild Dermatological', 'Risk: Severe Anaphylaxis']
}

# --- 3. Refined Inference Engine ---
def predict_pure_clinical_profile(drug_list):
    print(f"🩺 Purified Safety Profile: {' + '.join([d.upper() for d in drug_list])}")
    print("=" * 65)

    clean_drugs = ", ".join([d.strip().lower() for d in drug_list])
    x_input = tfidf.transform([clean_drugs]).astype(np.float32)
    x_tensor = torch.tensor(x_input.toarray().squeeze()).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(x_tensor)
        probs = torch.sigmoid(outputs).cpu().numpy()[0]

    found_any_risk = False
    for parent_class, sub_risks in HIERARCHY_MAP.items():
        category_risks = {}
        for i, risk in enumerate(PHENOTYPES):
            if risk in sub_risks and probs[i] > optimal_thresholds[i]:
                category_risks[risk] = probs[i]

        if category_risks:
            found_any_risk = True
            print(f"{parent_class}")
            print("-" * 40)
            sorted_risks = sorted(category_risks.items(), key=lambda x: x[1], reverse=True)
            for risk, prob in sorted_risks:
                risk_idx = PHENOTYPES.index(risk)
                thresh = optimal_thresholds[risk_idx]

                # FIXED: Relative danger margin calculation
                danger_margin = (prob - thresh) / (1.0 - thresh + 1e-5)

                bar_length = int(prob * 20)
                bar = "█" * bar_length + "░" * (20 - bar_length)

                # منطق ترکیبی بالینی و ریاضی
                if prob >= 0.75 or danger_margin > 0.60:
                    tag = "🔴 HIGH"
                elif prob >= 0.50 or danger_margin > 0.30:
                    tag = "🟠 MOD"
                else:
                    tag = "🟡 LOW"

                print(f"  ↳ {risk:<30} {tag} | {prob*100:4.1f}%  {bar}")
            print()

    if not found_any_risk:
        print("✅ No significant clinical risks detected above calculated thresholds.\n")
    print("=" * 65 + "\n")

# --- Final Tests ---
predict_pure_clinical_profile(["warfarin", "ibuprofen", "aspirin"])
predict_pure_clinical_profile(["atorvastatin", "amlodipine"])
predict_pure_clinical_profile(["fluoxetine", "alprazolam", "zolpidem"])

✅ Model, Thresholds, and Transformers Loaded Successfully!

🩺 Purified Safety Profile: WARFARIN + IBUPROFEN + ASPIRIN
🧠 Nervous System & Psych
----------------------------------------
  ↳ Risk: Vestibular/CNS           🟠 MOD | 56.1%  ███████████░░░░░░░░░

🩸 Bleeding & GI Disorders
----------------------------------------
  ↳ Risk: GI Bleeding              🔴 HIGH | 77.7%  ███████████████░░░░░
  ↳ Risk: Systemic Bleeding        🟡 LOW | 35.2%  ███████░░░░░░░░░░░░░


🩺 Purified Safety Profile: ATORVASTATIN + AMLODIPINE
🧠 Nervous System & Psych
----------------------------------------
  ↳ Risk: Vestibular/CNS           🟠 MOD | 58.6%  ███████████░░░░░░░░░

🫁 Respiratory Disorders
----------------------------------------
  ↳ Risk: Dyspnea/Bronchospasm     🟠 MOD | 53.4%  ██████████░░░░░░░░░░


🩺 Purified Safety Profile: FLUOXETINE + ALPRAZOLAM + ZOLPIDEM
🧠 Nervous System & Psych
----------------------------------------
  ↳ Risk: CNS Depression           🟠 MOD | 70.7%  ██████████████░░░░░░


